# Nghiệm thu Customer 360 Gold bằng Spark SQL

Notebook chạy sau khi Gold, segmentation và PII masking đã hoàn thành cho `2025-12-31` và `2026-01-01`. Logic kiểm tra nằm trong Spark SQL; Python chỉ hiển thị kết quả và dừng khi có `FAIL`.

In [1]:
try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

## 1. Current, history và masked mart

In [2]:
serving_contract = spark.sql("""
WITH metrics AS (
    SELECT 'current' AS table_name, COUNT(*) AS row_count,
           COUNT(DISTINCT customer_id) AS customer_count, MIN(cob_dt) AS min_cob_dt, MAX(cob_dt) AS max_cob_dt
    FROM lakehouse.gold.mart_customer_360
    UNION ALL
    SELECT 'history', COUNT(*), COUNT(DISTINCT customer_id), MIN(cob_dt), MAX(cob_dt)
    FROM lakehouse.gold.mart_customer_360_history
    UNION ALL
    SELECT 'masked', COUNT(*), COUNT(DISTINCT customer_id), MIN(cob_dt), MAX(cob_dt)
    FROM lakehouse.sandbox.mart_customer_360_masked
)
SELECT *,
       CASE
           WHEN table_name IN ('current', 'masked')
            AND row_count = 10000 AND customer_count = 10000
            AND min_cob_dt = DATE '2026-01-01' AND max_cob_dt = DATE '2026-01-01' THEN 'PASS'
           WHEN table_name = 'history'
            AND row_count = 20000 AND customer_count = 10000
            AND min_cob_dt = DATE '2025-12-31' AND max_cob_dt = DATE '2026-01-01' THEN 'PASS'
           ELSE 'FAIL'
       END AS status
FROM metrics
ORDER BY table_name
""")
serving_contract.show(truncate=False)
assert serving_contract.where("status = 'FAIL'").count() == 0, "Serving-date contract failed"

+----------+---------+--------------+----------+----------+------+
|table_name|row_count|customer_count|min_cob_dt|max_cob_dt|status|
+----------+---------+--------------+----------+----------+------+
|current   |10000    |10000         |2026-01-01|2026-01-01|PASS  |
|history   |20000    |10000         |2025-12-31|2026-01-01|PASS  |
|masked    |10000    |10000         |2026-01-01|2026-01-01|PASS  |
+----------+---------+--------------+----------+----------+------+



## 2. Uniqueness của current, history và campaign

In [3]:
uniqueness_checks = spark.sql("""
SELECT check_name, violations, CASE WHEN violations = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
    SELECT 'current_duplicate_customer' AS check_name, COUNT(*) AS violations
    FROM (SELECT customer_id FROM lakehouse.gold.mart_customer_360
          GROUP BY customer_id HAVING COUNT(*) > 1)
    UNION ALL
    SELECT 'history_duplicate_customer_date', COUNT(*)
    FROM (SELECT customer_id, cob_dt FROM lakehouse.gold.mart_customer_360_history
          GROUP BY customer_id, cob_dt HAVING COUNT(*) > 1)
    UNION ALL
    SELECT 'campaign_duplicate_customer_date', COUNT(*)
    FROM (SELECT customer_id, cob_dt FROM lakehouse.gold.campaign_target
          GROUP BY customer_id, cob_dt HAVING COUNT(*) > 1)
) checks
ORDER BY check_name
""")
uniqueness_checks.show(truncate=False)
assert uniqueness_checks.where("status = 'FAIL'").count() == 0, "Gold uniqueness failed"

+--------------------------------+----------+------+
|check_name                      |violations|status|
+--------------------------------+----------+------+
|campaign_duplicate_customer_date|0         |PASS  |
|current_duplicate_customer      |0         |PASS  |
|history_duplicate_customer_date |0         |PASS  |
+--------------------------------+----------+------+



## 3. Reconciliation AUM và giao dịch thẻ

Các nguồn 1:N được aggregate độc lập trước khi so với Gold.

In [4]:
balance_check = spark.sql("""
WITH account_agg AS (
    SELECT customer_id, SUM(CASE WHEN status = 'ACTIVE' THEN balance ELSE 0 END) AS account_balance
    FROM lakehouse.silver.dim_account
    WHERE DATE '2026-01-01' BETWEEN effective_from AND effective_to
    GROUP BY customer_id
),
deposit_agg AS (
    SELECT customer_id, SUM(CASE WHEN status = 'ACTIVE' THEN principal_amount ELSE 0 END) AS deposit_principal
    FROM lakehouse.silver.dim_deposit GROUP BY customer_id
),
loan_agg AS (
    SELECT customer_id, SUM(CASE WHEN loan_status = 'ACTIVE' THEN outstanding_balance ELSE 0 END) AS loan_outstanding
    FROM lakehouse.silver.dim_loan GROUP BY customer_id
),
expected AS (
    SELECT c.customer_id,
           COALESCE(a.account_balance, 0) + COALESCE(d.deposit_principal, 0) AS aum_total,
           COALESCE(l.loan_outstanding, 0) AS loan_outstanding
    FROM lakehouse.silver.dim_customer c
    LEFT JOIN account_agg a ON c.customer_id = a.customer_id
    LEFT JOIN deposit_agg d ON c.customer_id = d.customer_id
    LEFT JOIN loan_agg l ON c.customer_id = l.customer_id
    WHERE DATE '2026-01-01' BETWEEN c.effective_from AND c.effective_to
)
SELECT COUNT(*) AS violations, CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM expected e
JOIN lakehouse.gold.mart_customer_360 m ON e.customer_id = m.customer_id
WHERE m.aum_total <> CAST(e.aum_total AS DECIMAL(18,2))
   OR m.total_loan_outstanding <> CAST(e.loan_outstanding AS DECIMAL(18,2))
""")
balance_check.show(truncate=False)
assert balance_check.where("status = 'FAIL'").count() == 0, "AUM reconciliation failed"

+----------+------+
|violations|status|
+----------+------+
|0         |PASS  |
+----------+------+



In [5]:
card_check = spark.sql("""
WITH expected AS (
    SELECT customer_id, COUNT(txn_id) AS txn_count,
           CAST(ROUND(SUM(txn_amount), 2) AS DECIMAL(18,2)) AS txn_amount,
           COUNT(DISTINCT merchant_category) AS merchant_categories, MAX(txn_date) AS last_txn_date
    FROM lakehouse.silver.fact_card_txn
    WHERE CAST(txn_date AS DATE) BETWEEN DATE_ADD(DATE '2026-01-01', -30) AND DATE '2026-01-01'
      AND status = 'SUCCESS' AND txn_type NOT IN ('REFUND', 'REVERSAL')
    GROUP BY customer_id
)
SELECT COUNT(*) AS violations, CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM lakehouse.gold.customer_card_summary g
LEFT JOIN expected e ON g.customer_id = e.customer_id
WHERE g.cob_dt = DATE '2026-01-01'
  AND (g.total_card_txn_count_30d <> COALESCE(e.txn_count, 0)
    OR g.total_card_txn_amount_30d <> COALESCE(e.txn_amount, CAST(0 AS DECIMAL(18,2)))
    OR g.distinct_merchant_categories <> COALESCE(e.merchant_categories, 0)
    OR NOT (g.last_card_txn_date <=> e.last_txn_date))
""")
card_check.show(truncate=False)
assert card_check.where("status = 'FAIL'").count() == 0, "Card reconciliation failed"

+----------+------+
|violations|status|
+----------+------+
|0         |PASS  |
+----------+------+



## 4. RFM, campaign và flags

In [6]:
rfm_check = spark.sql("""
SELECT cob_dt, COUNT(*) AS customers, COUNT(DISTINCT rfm_segment) AS segment_count,
       MIN(r_score) AS min_r, MAX(r_score) AS max_r,
       MIN(f_score) AS min_f, MAX(f_score) AS max_f,
       MIN(m_score) AS min_m, MAX(m_score) AS max_m,
       MIN(rfm_score) AS min_rfm, MAX(rfm_score) AS max_rfm,
       CASE WHEN COUNT(*) = 10000 AND COUNT(DISTINCT rfm_segment) = 7
                  AND MIN(r_score) = 1 AND MAX(r_score) = 5
                  AND MIN(f_score) = 1 AND MAX(f_score) = 5
                  AND MIN(m_score) = 1 AND MAX(m_score) = 5
                  AND MIN(rfm_score) = 3 AND MAX(rfm_score) = 15
            THEN 'PASS' ELSE 'FAIL' END AS status
FROM lakehouse.gold.rfm_segment
GROUP BY cob_dt
ORDER BY cob_dt
""")
rfm_check.show(truncate=False)
spark.sql("""
SELECT cob_dt, rfm_segment, COUNT(*) AS customers
FROM lakehouse.gold.rfm_segment
GROUP BY cob_dt, rfm_segment
ORDER BY cob_dt, customers DESC
""").show(100, truncate=False)
assert rfm_check.where("status = 'FAIL'").count() == 0, "RFM contract failed"

+----------+---------+-------------+-----+-----+-----+-----+-----+-----+-------+-------+------+
|cob_dt    |customers|segment_count|min_r|max_r|min_f|max_f|min_m|max_m|min_rfm|max_rfm|status|
+----------+---------+-------------+-----+-----+-----+-----+-----+-----+-------+-------+------+
|2025-12-31|10000    |7            |1    |5    |1    |5    |1    |5    |3      |15     |PASS  |
|2026-01-01|10000    |7            |1    |5    |1    |5    |1    |5    |3      |15     |PASS  |
+----------+---------+-------------+-----+-----+-----+-----+-----+-----+-------+-------+------+

+----------+-------------------+---------+
|cob_dt    |rfm_segment        |customers|
+----------+-------------------+---------+
|2025-12-31|Champions          |2165     |
|2025-12-31|At Risk            |2139     |
|2025-12-31|Hibernating        |1923     |
|2025-12-31|Loyal Customers    |1773     |
|2025-12-31|New Customers      |893      |
|2025-12-31|Potential Loyalists|563      |
|2025-12-31|Lost               |544 

In [7]:
campaign_checks = spark.sql("""
SELECT check_name, violations, CASE WHEN violations = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
    SELECT 'campaign_duplicates' AS check_name, COUNT(*) AS violations
    FROM (SELECT customer_id, cob_dt FROM lakehouse.gold.campaign_target
          GROUP BY customer_id, cob_dt HAVING COUNT(*) > 1)
    UNION ALL
    SELECT 'invalid_cross_sell_flags', COUNT(*) FROM lakehouse.gold.cross_sell_segment
    WHERE no_credit_card NOT IN (0,1) OR no_deposit NOT IN (0,1) OR no_loan NOT IN (0,1)
    UNION ALL
    SELECT 'invalid_churn_flags', COUNT(*) FROM lakehouse.gold.churn_prediction
    WHERE is_churn_candidate NOT IN (0,1)
) checks
ORDER BY check_name
""")
campaign_checks.show(truncate=False)
assert campaign_checks.where("status = 'FAIL'").count() == 0, "Campaign/flag contract failed"

+------------------------+----------+------+
|check_name              |violations|status|
+------------------------+----------+------+
|campaign_duplicates     |0         |PASS  |
|invalid_churn_flags     |0         |PASS  |
|invalid_cross_sell_flags|0         |PASS  |
+------------------------+----------+------+



## 5. PII masking và ví dụ history

In [8]:
raw_pii = {'full_name', 'phone', 'email', 'cccd', 'address'}
masked_columns = set(spark.table('lakehouse.sandbox.mart_customer_360_masked').columns)
exposed_pii = sorted(raw_pii & masked_columns)
assert not exposed_pii, f'Raw PII exposed: {exposed_pii}'
print(f'Masked schema: PASS ({len(masked_columns)} columns, raw PII = 0)')

spark.sql("""
SELECT customer_id, cob_dt, primary_branch_code, customer_segment, aum_total, rfm_segment
FROM lakehouse.gold.mart_customer_360_history
WHERE customer_id = 10001
ORDER BY cob_dt
""").show(truncate=False)
print('CUSTOMER 360 GOLD ACCEPTANCE: PASS')

Masked schema: PASS (33 columns, raw PII = 0)
+-----------+----------+-------------------+----------------+-------------+-----------+
|customer_id|cob_dt    |primary_branch_code|customer_segment|aum_total    |rfm_segment|
+-----------+----------+-------------------+----------------+-------------+-----------+
|10001      |2025-12-31|BDG001             |VIP             |1359716000.00|Champions  |
|10001      |2026-01-01|BDG001             |PRIORITY        |511114000.00 |Champions  |
+-----------+----------+-------------------+----------------+-------------+-----------+

CUSTOMER 360 GOLD ACCEPTANCE: PASS
